# Exercise 2.1: Moving Existing Tables to Iceberg

In this exercise, you'll learn how to migrate existing data into Apache Iceberg tables using different strategies:
- **CTAS (Create Table As Select)**: Reserialization approach for any format
- **Metadata tables**: Inspect migrated table structure
- **Incremental loading**: Add new data over time

## Learning Objectives
- Migrate CSV data to Iceberg using CTAS
- Migrate Parquet data to Iceberg using CTAS
- Compare table layouts before and after migration
- Handle incremental data additions

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import urllib.request
import os

spark = SparkSession.builder \
    .appName("MovingExistingTables") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.migration")
print("Namespace 'migration' created!")

## Part 1: Migrate CSV Data Using CTAS

CTAS (Create Table As Select) is the most flexible migration approach. It works with any data source and allows transformation during migration.

### Create Sample CSV Data

In [ ]:
# Create a sample CSV file
csv_data = """customer_id,customer_name,signup_date,total_purchases
1,Alice Johnson,2023-01-15,45
2,Bob Smith,2023-02-20,12
3,Carol White,2023-03-10,78
4,David Brown,2023-04-05,34
5,Eve Davis,2023-05-12,56"""

# Write to local file
os.makedirs('/tmp/csv_data', exist_ok=True)
with open('/tmp/csv_data/customers.csv', 'w') as f:
    f.write(csv_data)

print("CSV file created at /tmp/csv_data/customers.csv")

### Read CSV File

In [ ]:
# Read CSV file
csv_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/tmp/csv_data/customers.csv")

# Show schema
print("CSV Schema:")
csv_df.printSchema()

# Show data
print("\nCSV Data:")
csv_df.show()

### Migrate to Iceberg

In [ ]:
# Create Iceberg table from CSV using DataFrame API
csv_df.writeTo("polaris.migration.customers_from_csv") \
    .using("iceberg") \
    .createOrReplace()

print("Table created from CSV!")

In [ ]:
# Query the new Iceberg table
spark.sql("SELECT * FROM polaris.migration.customers_from_csv ORDER BY customer_id").show()

**What happened?** 
- Spark read the CSV, parsed the data, and wrote new Parquet files
- This involved full data reserialization (reading + writing)
- The original CSV remains unchanged
- This is the most flexible but also most expensive migration method

### View Iceberg Metadata

In [ ]:
# View files in the table
spark.sql("""
SELECT 
    file_path,
    file_size_in_bytes,
    record_count
FROM polaris.migration.customers_from_csv.files
""").show(truncate=False)

## Part 2: Migrate Parquet Data Using CTAS

When you have existing Parquet files, you can also use CTAS to create Iceberg tables.

### Download NYC Taxi Parquet Data

In [ ]:
import boto3
from botocore.client import Config

# Configure MinIO client
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

bucket = 'warehouse'

# Download June 2023 taxi data
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-06.parquet"
local_file = "/tmp/yellow_tripdata_2023-06.parquet"

print("Downloading NYC Taxi data (June 2023)...")
print("This may take a few minutes...")
urllib.request.urlretrieve(url, local_file)
print(f"Downloaded to {local_file}")

# Upload to MinIO in a landing zone
s3_key = 'migration/landing/yellow_tripdata_2023-06.parquet'

print(f"Uploading to MinIO: s3://{bucket}/{s3_key}...")
s3_client.upload_file(local_file, bucket, s3_key)
print("Upload complete!")

### Examine the Parquet File

In [ ]:
# Read the Parquet file directly
parquet_df = spark.read.parquet(f"s3a://{bucket}/{s3_key}")

print("Schema:")
parquet_df.printSchema()

print(f"\nRow count: {parquet_df.count():,}")

print("\nSample data:")
parquet_df.show(5)

### Migrate Parquet to Iceberg Using CTAS

In [ ]:
# Create Iceberg table from Parquet using CTAS
print("Creating Iceberg table from Parquet...")

spark.sql(f"""
CREATE OR REPLACE TABLE polaris.migration.taxi_from_parquet
USING iceberg
AS SELECT * FROM parquet.`s3a://{bucket}/{s3_key}`
""")

print("Table created!")

In [ ]:
# Verify row count
iceberg_count = spark.sql("""
SELECT COUNT(*) as count 
FROM polaris.migration.taxi_from_parquet
""").collect()[0]['count']

print(f"Iceberg table row count: {iceberg_count:,}")

In [ ]:
# Sample query
spark.sql("""
SELECT 
    tpep_pickup_datetime,
    passenger_count,
    trip_distance,
    total_amount
FROM polaris.migration.taxi_from_parquet
LIMIT 5
""").show()

### Compare Original vs Iceberg Files

In [ ]:
# View Iceberg table files
print("Iceberg table files:")
spark.sql("""
SELECT 
    file_path,
    file_size_in_bytes / 1024 / 1024 as size_mb,
    record_count
FROM polaris.migration.taxi_from_parquet.files
""").show(truncate=False)

**Observation**: New Parquet files were created in the Iceberg warehouse location, separate from the original landing zone files.

## Part 3: Incremental Data Addition

Once you have an Iceberg table, you can add new data incrementally.

### Download Another Month of Data

In [ ]:
# Download July 2023 taxi data
url_july = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-07.parquet"
local_file_july = "/tmp/yellow_tripdata_2023-07.parquet"

print("Downloading NYC Taxi data (July 2023)...")
print("This may take a few minutes...")
urllib.request.urlretrieve(url_july, local_file_july)
print(f"Downloaded to {local_file_july}")

# Upload to MinIO
s3_key_july = 'migration/landing/yellow_tripdata_2023-07.parquet'
print(f"Uploading to MinIO: s3://{bucket}/{s3_key_july}...")
s3_client.upload_file(local_file_july, bucket, s3_key_july)
print("Upload complete!")

### Check Current Table State

In [ ]:
# Count rows before adding new data
before_count = spark.sql("""
SELECT COUNT(*) as count 
FROM polaris.migration.taxi_from_parquet
""").collect()[0]['count']

print(f"Rows before: {before_count:,}")

# Check file count
file_count_before = spark.sql("""
SELECT COUNT(*) as file_count
FROM polaris.migration.taxi_from_parquet.files
""").collect()[0]['file_count']

print(f"Files before: {file_count_before}")

### Add New Month Using INSERT INTO

In [ ]:
# Read July data
july_df = spark.read.parquet(f"s3a://{bucket}/{s3_key_july}")

print(f"July data has {july_df.count():,} rows")

# Append to Iceberg table
print("\nAppending July data to Iceberg table...")
july_df.writeTo("polaris.migration.taxi_from_parquet").append()

print("Data appended!")

### Verify the Addition

In [ ]:
# Count rows after adding new data
after_count = spark.sql("""
SELECT COUNT(*) as count 
FROM polaris.migration.taxi_from_parquet
""").collect()[0]['count']

print(f"Rows after: {after_count:,}")
print(f"Rows added: {after_count - before_count:,}")

# Check file count
file_count_after = spark.sql("""
SELECT COUNT(*) as file_count
FROM polaris.migration.taxi_from_parquet.files
""").collect()[0]['file_count']

print(f"\nFiles after: {file_count_after}")
print(f"Files added: {file_count_after - file_count_before}")

In [ ]:
# View all files now in the table
spark.sql("""
SELECT 
    file_path,
    file_size_in_bytes / 1024 / 1024 as size_mb,
    record_count
FROM polaris.migration.taxi_from_parquet.files
ORDER BY file_path
""").show(truncate=False)

## Part 4: View Migration History

In [ ]:
# View snapshots created during migration
print("Migration history (snapshots):")
spark.sql("""
SELECT 
    snapshot_id,
    committed_at,
    operation,
    summary['added-records'] as records_added
FROM polaris.migration.taxi_from_parquet.snapshots
ORDER BY committed_at
""").show(truncate=False)

## Part 5: Query Performance Benefits

### Create Partitioned Version for Comparison

In [ ]:
# Create a partitioned version
print("Creating partitioned Iceberg table...")

spark.sql(f"""
CREATE OR REPLACE TABLE polaris.migration.taxi_partitioned
USING iceberg
PARTITIONED BY (days(tpep_pickup_datetime))
AS SELECT * FROM parquet.`s3a://{bucket}/migration/landing/yellow_tripdata_2023-06.parquet`
""")

print("Partitioned table created!")

In [ ]:
# View partitions
print("Partitions in table:")
spark.sql("""
SELECT 
    partition,
    COUNT(*) as file_count,
    SUM(record_count) as records
FROM polaris.migration.taxi_partitioned.files
GROUP BY partition
ORDER BY partition
LIMIT 10
""").show(truncate=False)

## Migration Strategy Summary

### When to Use Each Approach

| Strategy | Use Case | Data Copy? | Speed | Flexibility |
|----------|----------|------------|-------|-------------|
| **CTAS from CSV/JSON** | Non-Parquet formats | Yes | Slow | High - can transform |
| **CTAS from Parquet** | Existing Parquet with transformation | Yes | Moderate | High - can repartition |
| **INSERT INTO** | Incremental additions | Yes | Moderate | Medium |

**Key Points:**
- CTAS is the most flexible approach - works with any source
- You can transform, filter, and repartition during migration
- New data files are created in Iceberg format
- Original source files remain unchanged

## Key Takeaways

1. **CTAS for flexibility**: Works with any data source and allows transformations
2. **Plan for partitioning**: Consider partitioning strategy during migration
3. **Incremental loading**: Use INSERT INTO or DataFrame.append() for new data
4. **Metadata tracking**: Every migration operation creates a snapshot
5. **Test before committing**: Validate migrated data before using in production

## Real-World Migration Workflow

1. **Analyze source data**: Understand schema, volume, and query patterns
2. **Choose partition strategy**: Based on common queries
3. **Test migration on subset**: Validate approach with sample data
4. **Migrate historical data**: Use CTAS for bulk migration
5. **Set up incremental loading**: Automate adding new data
6. **Validate and monitor**: Ensure queries work and performance is good

## Challenge Exercise

1. Create your own CSV dataset
2. Migrate it to Iceberg with a partition strategy
3. Add more data incrementally
4. Query the table and verify correctness
5. Use metadata tables to understand the structure

## Cleanup

In [ ]:
# Optional: Drop the tables if you want to start fresh
# spark.sql("DROP TABLE IF EXISTS polaris.migration.customers_from_csv")
# spark.sql("DROP TABLE IF EXISTS polaris.migration.taxi_from_parquet")
# spark.sql("DROP TABLE IF EXISTS polaris.migration.taxi_partitioned")
# print("Tables dropped!")